<a href="https://colab.research.google.com/github/akim201104-creator/UTS_Zidane_Zamill_Hakim_14022300067_6B-BIS./blob/main/UTS_Zidane_Zamil_Hakim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'id.go.bps.allstats',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
import pandas as pd
from transformers import pipeline

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(filename)

display(df.head())

In [ ]:
# Initialize the sentiment analysis pipeline with the specified model
sentiment_analyzer = pipeline('sentiment-analysis', model='w11wo/indonesian-roberta-base-prdect-id')

In [ ]:
# Apply sentiment analysis to the 'content' column
sentiments = sentiment_analyzer(df['content'].tolist())

# Extract labels and scores
df['sentiment_label'] = [s['label'] for s in sentiments]
df['sentiment_score'] = [s['score'] for s in sentiments]

# Display the DataFrame with sentiment results
display(df[['content', 'sentiment_label', 'sentiment_score']].head())

In [ ]:
# Save the DataFrame with sentiment results back to the CSV file
df.to_csv(filename, index=False, encoding='utf-8')

print(f"File CSV '{filename}' berhasil diperbarui dengan hasil sentimen.")

In [ ]:
# Define a mapping for positive and negative sentiments
def map_to_binary_sentiment(label):
    if label in ['Happy', 'Love']:
        return 'Positive'
    elif label in ['Fear', 'Anger', 'Sadness']:
        return 'Negative'
    else:
        return 'Neutral' # Handle any other labels if they exist

# Apply the mapping to create a new binary sentiment column
df['binary_sentiment'] = df['sentiment_label'].apply(map_to_binary_sentiment)

# Display the DataFrame with the new binary sentiment results
display(df[['content', 'sentiment_label', 'binary_sentiment', 'sentiment_score']].head())

In [ ]:
new_filename = 'ulasan_google_play_binary_sentiment.csv'

# Save the DataFrame with binary sentiment results to a new CSV file
df.to_csv(new_filename, index=False, encoding='utf-8')

print(f"File CSV '{new_filename}' berhasil diperbarui dengan hasil sentimen biner.")

In [ ]:
# Count the occurrences of each binary sentiment label
binary_sentiment_counts = df['binary_sentiment'].value_counts().reset_index()
binary_sentiment_counts.columns = ['binary_sentiment', 'count']

# Create a bar plot of the binary sentiment distribution
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.barplot(x='binary_sentiment', y='count', data=binary_sentiment_counts, hue='binary_sentiment', palette='coolwarm', legend=False)
plt.title('Distribusi Sentimen Biner Ulasan Google Play')
plt.xlabel('Label Sentimen Biner')
plt.ylabel('Jumlah Ulasan')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count the occurrences of each sentiment label
sentiment_counts = df['sentiment_label'].value_counts().reset_index()
sentiment_counts.columns = ['sentiment_label', 'count']

# Create a bar plot of the sentiment distribution
plt.figure(figsize=(8, 6))
sns.barplot(x='sentiment_label', y='count', data=sentiment_counts, hue='sentiment_label', palette='viridis', legend=False)
plt.title('Distribusi Sentimen Ulasan Google Play')
plt.xlabel('Label Sentimen')
plt.ylabel('Jumlah Ulasan')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Analisis Sentimen Biner Langsung

Untuk melakukan klasifikasi langsung menjadi 'Positif' atau 'Negatif' tanpa melewati kategori emosi perantara, kita akan menggunakan model analisis sentimen biner Indonesia yang berbeda.

In [ ]:
# Initialize a new sentiment analysis pipeline with a binary classification model
binary_sentiment_analyzer = pipeline('sentiment-analysis', model='w11wo/indonesian-roberta-base-sentiment-classifier')

# Apply binary sentiment analysis to the 'content' column
binary_sentiments = binary_sentiment_analyzer(df['content'].tolist())

# Extract labels and scores for binary sentiment
df['direct_binary_sentiment_label'] = [s['label'] for s in binary_sentiments]
df['direct_binary_sentiment_score'] = [s['score'] for s in binary_sentiments]

# Display the DataFrame with the new direct binary sentiment results
display(df[['content', 'direct_binary_sentiment_label', 'direct_binary_sentiment_score']].head())

In [ ]:
new_filename_direct_binary = 'ulasan_google_play_direct_binary_sentiment.csv'

# Save the DataFrame with direct binary sentiment results to a new CSV file
df.to_csv(new_filename_direct_binary, index=False, encoding='utf-8')

print(f"File CSV '{new_filename_direct_binary}' berhasil diperbarui dengan hasil sentimen biner langsung.")

In [ ]:
# Count the occurrences of each direct binary sentiment label
direct_binary_sentiment_counts = df['direct_binary_sentiment_label'].value_counts().reset_index()
direct_binary_sentiment_counts.columns = ['direct_binary_sentiment_label', 'count']

# Create a bar plot of the direct binary sentiment distribution
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.barplot(x='direct_binary_sentiment_label', y='count', data=direct_binary_sentiment_counts, hue='direct_binary_sentiment_label', palette='viridis', legend=False)
plt.title('Distribusi Sentimen Biner Langsung Ulasan Google Play')
plt.xlabel('Label Sentimen Biner')
plt.ylabel('Jumlah Ulasan')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()